# Exercise 1A: Build and Explore Price Data
**BUSI 722: Data-Driven Finance II**

Query the Rice Data Portal API to build a monthly dataset for all stocks
from January 2023 through January 2026.

In [1]:
import requests
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
TOKEN = os.getenv("RICE_ACCESS_TOKEN")
API_URL = "https://data-portal.rice-business.org/api/query"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def query_api(sql):
    resp = requests.post(API_URL, json={"query": sql}, headers=HEADERS)
    resp.raise_for_status()
    data = resp.json()
    return pd.DataFrame(data["data"], columns=data["columns"])

print("API connection configured.")

API connection configured.


## Step 1: Fetch End-of-Month Prices from SEP

We fetch data from January 2021 onward to have enough history for computing
12-month moving averages and 13-month momentum at the start of our
January 2023 analysis window. Queries are split by year to avoid API timeouts.

In [2]:
print("Fetching end-of-month prices from SEP...")
frames = []
year_ranges = [
    ("2021-01-01", "2022-01-01"),
    ("2022-01-01", "2023-01-01"),
    ("2023-01-01", "2024-01-01"),
    ("2024-01-01", "2025-01-01"),
    ("2025-01-01", "2026-01-01"),
    ("2026-01-01", "2026-02-01"),
]

for start, end in year_ranges:
    sql = f"""
    WITH month_ends AS (
      SELECT a.ticker, a.date::DATE as date, a.close, a.closeadj,
             ROW_NUMBER() OVER (
               PARTITION BY a.ticker, DATE_TRUNC('month', a.date::DATE)
               ORDER BY a.date::DATE DESC
             ) as rn
      FROM sep a
      WHERE a.date::DATE >= '{start}' AND a.date::DATE < '{end}'
    )
    SELECT ticker, CAST(date AS VARCHAR) as date, close, closeadj
    FROM month_ends WHERE rn = 1
    ORDER BY ticker, date
    """
    chunk = query_api(sql)
    frames.append(chunk)
    print(f"  {start[:7]} to {end[:7]}: {len(chunk):,} rows")

sep = pd.concat(frames, ignore_index=True)
sep["date"] = pd.to_datetime(sep["date"])
sep["month"] = sep["date"].dt.to_period("M").astype(str)
sep["close"] = pd.to_numeric(sep["close"], errors="coerce")
sep["closeadj"] = pd.to_numeric(sep["closeadj"], errors="coerce")
print()
print(f"Total SEP: {len(sep):,} rows, {sep['ticker'].nunique()} tickers")
print(f"Month range: {sep['month'].min()} to {sep['month'].max()}")

Fetching end-of-month prices from SEP...


  2021-01 to 2022-01: 59,913 rows


  2022-01 to 2023-01: 62,585 rows


  2023-01 to 2024-01: 57,985 rows


  2024-01 to 2025-01: 53,968 rows


  2025-01 to 2026-01: 51,983 rows


  2026-01 to 2026-02: 4,326 rows

Total SEP: 290,760 rows, 6322 tickers
Month range: 2021-01 to 2026-01


## Step 2: Fetch End-of-Month Market Cap from DAILY

In [3]:
print("Fetching end-of-month market cap from DAILY...")
frames = []
for start, end in year_ranges:
    sql = f"""
    WITH month_ends AS (
      SELECT d.ticker, d.date::DATE as date, d.marketcap,
             ROW_NUMBER() OVER (
               PARTITION BY d.ticker, DATE_TRUNC('month', d.date::DATE)
               ORDER BY d.date::DATE DESC
             ) as rn
      FROM daily d
      WHERE d.date::DATE >= '{start}' AND d.date::DATE < '{end}'
    )
    SELECT ticker, CAST(date AS VARCHAR) as date, marketcap
    FROM month_ends WHERE rn = 1
    ORDER BY ticker, date
    """
    chunk = query_api(sql)
    frames.append(chunk)
    print(f"  {start[:7]} to {end[:7]}: {len(chunk):,} rows")

daily = pd.concat(frames, ignore_index=True)
daily["date"] = pd.to_datetime(daily["date"])
daily["month"] = daily["date"].dt.to_period("M").astype(str)
daily["marketcap"] = pd.to_numeric(daily["marketcap"], errors="coerce")
print()
print(f"Total DAILY: {len(daily):,} rows, {daily['ticker'].nunique()} tickers")

Fetching end-of-month market cap from DAILY...


  2021-01 to 2022-01: 59,892 rows


  2022-01 to 2023-01: 62,561 rows


  2023-01 to 2024-01: 57,971 rows


  2024-01 to 2025-01: 53,961 rows


  2025-01 to 2026-01: 51,981 rows


  2026-01 to 2026-02: 4,326 rows

Total DAILY: 290,692 rows, 6322 tickers


## Step 3: Fetch Sector and Industry from TICKERS

In [4]:
print("Fetching sector and industry from TICKERS...")
tickers_df = query_api("SELECT ticker, sector, industry FROM tickers")
print(f"Total tickers: {len(tickers_df):,}")
tickers_df.head()

Fetching sector and industry from TICKERS...


Total tickers: 15,398


,ticker,sector,industry
0,A,Healthcare,Diagnostics & Research
1,AA,Basic Materials,Aluminum
2,AAAB,Financial Services,Banks - Regional
3,AABC,Financial Services,Banks - Regional
4,AAC,Industrials,Shell Companies


## Step 4: Merge All Data

In [5]:
df = sep.merge(daily[["ticker", "month", "marketcap"]], on=["ticker", "month"], how="left")
df = df.merge(tickers_df, on="ticker", how="left")
df = df.sort_values(["ticker", "month"]).reset_index(drop=True)
print(f"Merged dataset: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"Columns: {list(df.columns)}")

Merged dataset: 290,760 rows, 6322 tickers
Columns: ['ticker', 'date', 'close', 'closeadj', 'month', 'marketcap', 'sector', 'industry']


## Step 5: Compute Monthly Returns

$$r_t = \frac{\text{closeadj}_t}{\text{closeadj}_{t-1}} - 1$$

In [6]:
df["return"] = df.groupby("ticker")["closeadj"].pct_change()
print(f"Returns computed. Non-null: {df['return'].notna().sum():,}")

Returns computed. Non-null: 284,438


## Step 6: Compute Momentum

Cumulative return from month $t{-}13$ to month $t{-}2$ (skipping the most recent month):

$$\text{momentum}_t = \frac{\text{closeadj}_{t-2}}{\text{closeadj}_{t-13}} - 1$$

In [7]:
df["momentum"] = (
    df.groupby("ticker")["closeadj"].shift(2)
    / df.groupby("ticker")["closeadj"].shift(13)
    - 1
)
print(f"Momentum computed. Non-null: {df['momentum'].notna().sum():,}")

Momentum computed. Non-null: 212,428


## Step 7: Compute Lagged Return

The prior month's return.

In [8]:
df["lag_return"] = df.groupby("ticker")["return"].shift(1)
print(f"Lagged return computed. Non-null: {df['lag_return'].notna().sum():,}")

Lagged return computed. Non-null: 278,169


## Step 8: Apply Filters

1. Penny-stock filter: drop rows where `close` < \$5
2. Drop rows with missing `return`, `momentum`, or `marketcap`

In [9]:
print(f"Before filters: {len(df):,} rows")

df = df[df["close"] >= 5.0].copy()
print(f"After penny stock filter (close >= $5): {len(df):,} rows")

df = df.dropna(subset=["return", "momentum", "marketcap"])
print(f"After dropping missing return/momentum/marketcap: {len(df):,} rows")

df = df.sort_values(["ticker", "month"]).reset_index(drop=True)

Before filters: 290,760 rows
After penny stock filter (close >= $5): 242,799 rows
After dropping missing return/momentum/marketcap: 171,590 rows


## Step 9: Summary Statistics

Report for the analysis window: January 2023 through January 2026.

In [10]:
df_analysis = df[(df["month"] >= "2023-01") & (df["month"] <= "2026-01")].copy()

print("Analysis window (Jan 2023 -- Jan 2026):")
print(f"  Total rows:     {len(df_analysis):,}")
print(f"  Unique tickers: {df_analysis['ticker'].nunique()}")
print(f"  Date range:     {df_analysis['month'].min()} to {df_analysis['month'].max()}")

Analysis window (Jan 2023 -- Jan 2026):
  Total rows:     127,863
  Unique tickers: 4758
  Date range:     2023-01 to 2026-01


In [11]:
months = sorted(df_analysis["month"].unique())
n_months = len(months)
ticker_counts = df_analysis.groupby("ticker")["month"].nunique()
full_tickers = (ticker_counts == n_months).sum()

print(f"Months in sample: {n_months}")
print(f"Tickers with data in every month: {full_tickers}")

Months in sample: 37
Tickers with data in every month: 2393


In [12]:
print("Summary statistics (Jan 2023 -- Jan 2026):")
for col in ["return", "momentum", "marketcap"]:
    s = df_analysis[col]
    print(f"  {col}:")
    print(f"    Mean:   {s.mean():.4f}")
    print(f"    Median: {s.median():.4f}")
    print(f"    Std:    {s.std():.4f}")
    print(f"    Min:    {s.min():.4f}")
    print(f"    Max:    {s.max():.4f}")
    print()

Summary statistics (Jan 2023 -- Jan 2026):
  return:
    Mean:   0.0171
    Median: 0.0037
    Std:    0.2550
    Min:    -0.9799
    Max:    22.5714

  momentum:
    Mean:   0.0854
    Median: 0.0301
    Std:    0.7775
    Min:    -1.0000
    Max:    45.6800

  marketcap:
    Mean:   16489.1986
    Median: 1461.2000
    Std:    115295.6359
    Min:    0.0000
    Max:    4920507.0000



In [13]:
df.to_parquet("exercise01_data.parquet", index=False)
print(f"Saved exercise01_data.parquet: {len(df):,} rows")
print(f"  Analysis window rows: {len(df_analysis):,}")
print(f"  Pre-2023 rows (for MA history): {len(df) - len(df_analysis):,}")

Saved exercise01_data.parquet: 171,590 rows
  Analysis window rows: 127,863
  Pre-2023 rows (for MA history): 43,727
